# Lab 11: Home exercise

* Student: Phạm Trần Minh Trí  
* ID: 2313622  
* Dataset: https://huggingface.co/datasets/Salesforce/wikitext

Import libraries

In [1]:
%pip install lightning

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.callbacks import ModelCheckpoint

## 1. Load data

Download dataset

In [3]:
dataset = load_dataset("wikitext", "wikitext-103-v1")

## 2. Preprocess data

Hyperparameters

In [ ]:
MAX_SEQ_LEN = 256  # sequence length for each data sample
BATCH_SIZE = 16  # number of sample in each batch for training
LEARNING_RATE = 1e-3
SCHEDULER_T0 = 2000  # T_0 for cosine scheduler 
MAX_EPOCHS = 2  # number of train epoch
NUM_WORKERS = 8  # number of cpu core for loading data
VAL_STEPS = 50  # number of training batches before validation and checkpoint
PATIENCE = 10  # number of val check before early stopping

Using GPT2 tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token  # gpt2 does not have pad token

Tokenize the dataset

In [6]:
tokenized_dataset = dataset.map(
    lambda example: tokenizer(example["text"]), batched=True
)

Turn to chunks of 256 tokens

In [7]:
def group_texts(examples):
    keys = ["input_ids", "attention_mask"]
    concat_examples = {}
    for key in keys:
        concat_examples[key] = sum(examples[key], [])

    total_len = len(concat_examples["input_ids"])

    result_data = {}
    for key, value in concat_examples.items():
        result_data[key] = [
            value[i : i + MAX_SEQ_LEN] for i in range(0, total_len, MAX_SEQ_LEN)
        ]

    return result_data


column_names = tokenized_dataset["train"].column_names
grouped_dataset = tokenized_dataset.map(
    function=group_texts, batched=True, remove_columns=column_names
)

Set pytorch format

In [8]:
grouped_dataset.set_format("torch")

Add data collator for creating batch

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Create dataloader

In [10]:
train_loader = DataLoader(
    dataset=grouped_dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    dataset=grouped_dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    dataset=grouped_dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)

## 3. Create model

Positional encoding

In [ ]:
class CausalLanguageModel(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        max_seq_len: int = MAX_SEQ_LEN,
        n_heads: int = 8,
        n_layers: int = 6,
    ):
        super().__init__()
        self.vocab_size = vocab_size

        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=d_model
        )
        self.pos_embedding = nn.Embedding(
            num_embeddings=max_seq_len, embedding_dim=d_model
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, batch_first=True, norm_first=True
            ),
            num_layers=n_layers,
            enable_nested_tensor=False,
        )
        self.fc = nn.Linear(in_features=d_model, out_features=vocab_size)

    def forward(self, input_ids, attention_mask):
        """
        - input_ids: (batch_size, seq_len)
        - attention_mask: (batch_size, seq_len)
        """
        batch_size, seq_len = input_ids.size()

        positions = torch.arange(0, seq_len, device=self.device)
        x = self.token_embedding(input_ids) + self.pos_embedding(positions)

        causal_mask = nn.Transformer.generate_square_subsequent_mask(
            sz=seq_len, device=self.device, dtype=torch.bool
        )
        padding_mask = attention_mask == 0
        x = self.transformer(
            src=x, mask=causal_mask, src_key_padding_mask=padding_mask, is_causal=True
        )

        logits = self.fc(x)
        return logits

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"][:, :-1]  # (batch_size, seq_len)
        attention_mask = batch["attention_mask"][:, :-1]  # (batch_size, seq_len)
        labels: torch.Tensor = batch["input_ids"][:, 1:]  # (batch_size, seq_len)

        logits: torch.Tensor = self(
            input_ids, attention_mask
        )  # (batch_size, seq_len, vocab_size)

        flatten_logits = logits.reshape(
            -1, self.vocab_size
        )  # (batch_size * seq_len, vocab_size)
        flatten_labels = labels.reshape(-1)  # (batch_size * seq_len)

        loss = F.cross_entropy(flatten_logits, flatten_labels)
        self.log("train_loss", loss)

        accuracy = (flatten_logits.argmax(1) == flatten_labels).float().mean()
        self.log("train_acc", accuracy)

        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"][:, :-1]  # (batch_size, seq_len)
        attention_mask = batch["attention_mask"][:, :-1]  # (batch_size, seq_len)
        labels: torch.Tensor = batch["input_ids"][:, 1:]  # (batch_size, seq_len)

        logits: torch.Tensor = self(
            input_ids, attention_mask
        )  # (batch_size, seq_len, vocab_size)

        flatten_logits = logits.reshape(
            -1, self.vocab_size
        )  # (batch_size * seq_len, vocab_size)
        flatten_labels = labels.reshape(-1)  # (batch_size * seq_len)

        loss = F.cross_entropy(flatten_logits, flatten_labels)
        self.log("val_loss", loss)

        accuracy = (flatten_logits.argmax(1) == flatten_labels).float().mean()
        self.log("val_acc", accuracy)

    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"][:, :-1]  # (batch_size, seq_len)
        attention_mask = batch["attention_mask"][:, :-1]  # (batch_size, seq_len)
        labels: torch.Tensor = batch["input_ids"][:, 1:]  # (batch_size, seq_len)

        logits: torch.Tensor = self(
            input_ids, attention_mask
        )  # (batch_size, seq_len, vocab_size)

        flatten_logits = logits.reshape(
            -1, self.vocab_size
        )  # (batch_size * seq_len, vocab_size)
        flatten_labels = labels.reshape(-1)  # (batch_size * seq_len)

        loss = F.cross_entropy(flatten_logits, flatten_labels)
        self.log("test_loss", loss)

        accuracy = (flatten_logits.argmax(1) == flatten_labels).float().mean()
        self.log("test_acc", accuracy)

    def configure_optimizers(self):
        optimizer = AdamW(params=self.parameters(), lr=LEARNING_RATE)
        scheduler = CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=SCHEDULER_T0)
        return [optimizer], [scheduler]
        


model = CausalLanguageModel(vocab_size=tokenizer.vocab_size)

## 4. Training

In [ ]:
trainer = L.Trainer(
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=PATIENCE, verbose=True),
        ModelCheckpoint(monitor="val_loss", every_n_train_steps=VAL_STEPS),
    ],
    max_epochs=MAX_EPOCHS,
    limit_train_batches=500,
    val_check_interval=VAL_STEPS,
)
trainer.fit(model, train_loader, val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ token_embedding │ Embedding          │ 25.7 M │ train │     0 │
│ 1 │ pos_embedding   │ Embedding          │  131 K │ train │     0 │
│ 2 │ transformer     │ TransformerEncoder │ 18.9 M │ train │     0 │
│ 3 │ fc              │ Linear             │ 25.8 M │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 70.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 70.6 M                                                                                               
Total estimated model params size (MB): 282                                                                        
Modules in train mode: 65                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/minht/miniconda3/envs/dl/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

Metric val_loss improved by 0.015 >= min_delta = 0.0. New best score: 7.419


Metric val_loss improved by 0.014 >= min_delta = 0.0. New best score: 7.404


Metric val_loss improved by 0.004 >= min_delta = 0.0. New best score: 7.400


Metric val_loss improved by 0.003 >= min_delta = 0.0. New best score: 7.397


Metric val_loss improved by 0.015 >= min_delta = 0.0. New best score: 7.382


Test the model on test set after training complete

In [ ]:
trainer.test(dataloaders=test_loader)

/home/minht/miniconda3/envs/dl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /home/minht/projects/NLP/lab/lab_11/epoch=2-step=300.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/minht/projects/NLP/lab/lab_11/epoch=2-step=300.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.04944509640336037    │
│         test_loss         │     7.414365291595459     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 7.414365291595459, 'test_acc': 0.04944509640336037}]